# 01 · Pipeline pilot (Qwen3.5-4B, bf16 LoRA SFT)

**Goal:** validate the whole flow on a cheap model before training the real 35B-A3B:
data → training → LoRA → GGUF → local LM Studio → `eval/run_eval.py`.

**The pilot is not about improving scores.** The only question is whether every step of the pipeline runs without errors.

- **Runtime:** L4 GPU (*Runtime → Change runtime type → L4*). ≈1.7 CU/hour, 5–10 CU in total.
- **First:** add `HF_TOKEN` (write access) in Colab's *Secrets* (🔑) panel and enable notebook access.
- **Checkpoints** are written to Google Drive. If the session drops, rerun the notebook from the top and training resumes.

In [ ]:
# Versions are not pinned: Unsloth picks packages that match Colab's torch (Qwen3.5 needs transformers v5)
!pip install -q -U unsloth unsloth_zoo
!pip install -q "transformers==5.2.0"
!pip install -q -U nvidia-nccl-cu13      # torch cu13 builds need a newer NCCL (ncclDevCommDestroy error)
!pip list 2>/dev/null | grep -iE "^(torch|torchvision|triton|xformers|nvidia-nccl-cu1[23]|unsloth|unsloth_zoo|transformers|trl|peft) "

In [ ]:
# Check: do torch + torchvision compiled ops load? If this fails, look at the pip list above
import torch, torchvision.ops
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Settings

In [ ]:
MODEL_NAME = "unsloth/Qwen3.5-4B"
RUN_NAME = "pilot-qwen35-4b"
MAX_SEQ_LENGTH = 4096

# Data mix (open-r1/Mixture-of-Thoughts, Apache 2.0): DeepSeek-R1 reasoning traces
MIX = {"math": 500, "code": 500, "science": 250}
MAX_STEPS = 150            # pilot: kept short
LEARNING_RATE = 1e-4
LORA_R = 16

HF_REPO = None             # e.g. "your_username/pilot-qwen35-4b" (private). None = no upload to the Hub.
DRIVE_DIR = f"/content/drive/MyDrive/colab-ai/{RUN_NAME}"

In [ ]:
import os
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
login(token=userdata.get("HF_TOKEN"))

!nvidia-smi --query-gpu=name,memory.total --format=csv

## Model + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, processor = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,      # QLoRA is not recommended for Qwen3.5
    load_in_16bit = True,
    full_finetuning = False,
)
# Qwen3.5 is multimodal; text training uses the underlying tokenizer
tokenizer = getattr(processor, "tokenizer", processor)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_R,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ_LENGTH,
)

## Data

- Examples are streamed from each subset and rendered with the model's **own chat template**.
- Examples longer than `MAX_SEQ_LENGTH` are skipped, not truncated. A cut-off reasoning trace hurts the model.
- Unsloth recommends at least 75% reasoning examples to preserve reasoning ability. All examples here contain reasoning.

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets

def take_fitting(subset, n):
    stream = load_dataset("open-r1/Mixture-of-Thoughts", subset, split="train", streaming=True)
    rows, seen = [], 0
    for ex in stream.shuffle(seed=3407, buffer_size=5_000):
        seen += 1
        text = tokenizer.apply_chat_template(ex["messages"], tokenize=False)
        if len(tokenizer(text, add_special_tokens=False)["input_ids"]) <= MAX_SEQ_LENGTH:
            rows.append({"text": text, "subset": subset})
            if len(rows) == n:
                break
    print(f"{subset}: {len(rows)}/{seen} examples fit the length limit")
    return Dataset.from_list(rows)

dataset = concatenate_datasets([take_fitting(s, n) for s, n in MIX.items()]).shuffle(seed=3407)
split = dataset.train_test_split(test_size=0.05, seed=3407)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds)

In [ ]:
# Check: is the template right? A <think> block and <|im_start|>assistant should be visible
print(train_ds[0]["text"][:1500])

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        max_steps = MAX_STEPS,
        learning_rate = LEARNING_RATE,
        lr_scheduler_type = "cosine",
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 25,
        save_total_limit = 2,
        output_dir = f"{DRIVE_DIR}/checkpoints",
        optim = "adamw_8bit",
        seed = 3407,
        dataset_num_proc = 1,
        report_to = "none",
    ),
)
# Loss is computed on the assistant response only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# Check: is masking right? Only the assistant part should remain (the rest is -100)
sample = trainer.train_dataset[0]
labels = [t for t in sample["labels"] if t != -100]
print(f"Trained tokens: {len(labels)}/{len(sample['labels'])}")
print(tokenizer.decode(labels)[:500])

In [ ]:
import glob
resume = bool(glob.glob(f"{DRIVE_DIR}/checkpoints/checkpoint-*"))
print("Resume from checkpoint:", resume)
stats = trainer.train(resume_from_checkpoint=resume)
print(stats.metrics)

## Quick check

This is not a benchmark, only a quick look to make sure the model is not broken.
The real comparison is done locally with `eval/run_eval.py`.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
prompts = [
    "Compute the exact value of the sum $\\sum_{n=1}^{\\infty} \\frac{1}{n(n+2)}$. Put the final answer in \\boxed{}.",
    "Write a Python function that returns the n-th Fibonacci number in O(log n) time.",
]
for p in prompts:
    text = tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    model.generate(**inputs, max_new_tokens=1500, temperature=0.6, top_p=0.95, top_k=20,
                   streamer=TextStreamer(tokenizer, skip_prompt=True))
    print("\n" + "=" * 80)

## Save: LoRA + GGUF

In [ ]:
model.save_pretrained(f"{DRIVE_DIR}/lora")
tokenizer.save_pretrained(f"{DRIVE_DIR}/lora")

# GGUF export builds llama.cpp; the first run can take ~10 min
model.save_pretrained_gguf(f"{DRIVE_DIR}/gguf", tokenizer, quantization_method = "q4_k_m")
!ls -lh {DRIVE_DIR}/gguf

In [ ]:
if HF_REPO:
    model.push_to_hub(HF_REPO + "-lora", private=True)
    tokenizer.push_to_hub(HF_REPO + "-lora", private=True)
    model.push_to_hub_gguf(HF_REPO + "-GGUF", tokenizer, quantization_method = "q4_k_m", private=True)

## Finish: stop spending CU
The files are on Drive. This cell releases the GPU.

In [ ]:
from google.colab import runtime
runtime.unassign()

## Local validation
1. Download the `.gguf` file from the `gguf/` folder on Drive and put it in LM Studio's models folder (e.g. `~/.lmstudio/models/local/pilot-qwen35-4b/`).
2. For a comparison, load the base `Qwen3.5-4B` first and then the pilot model in LM Studio, and run the same eval for both:
   ```
   python eval/run_eval.py --tasks eval/tasks/personal.jsonl eval/tasks/math500.jsonl --label pilot-base --limit 20
   python eval/run_eval.py --tasks eval/tasks/personal.jsonl eval/tasks/math500.jsonl --label pilot-sft --limit 20
   ```
3. Checklist:
   - [ ] The GGUF loads in LM Studio
   - [ ] The thinking section is separated correctly
   - [ ] The tool-call task works
   - [ ] Scores do not drop clearly below the base model